# Comparing Prompt Optimizers

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/quickstart/compare-optimizers)

Run three prompt searches on the same two routing tickets and compare them with the original prompt on two held-out tickets.

## Set up Python

Use a fresh Python 3.12 kernel. The key cell uses existing environment variables or prompts without displaying your input.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "agent-learning-kit==0.3.7"],
    check=True,
);

In [ ]:
import os
from getpass import getpass

for name in ['OPENAI_API_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ").strip()
        if not value:
            raise ValueError(f"{name} is required")
        os.environ[name] = value
        del value

## Compare three ways to use feedback

ProTeGi uses critiques, GEPA reflects on scored responses, and PromptWizard mutates and refines candidates. Each search gets the same exact-match rule and training tickets.

In [ ]:
from fi.evals.metrics import Equals
from fi.opt.base import Evaluator
from fi.opt.datamappers import BasicDataMapper
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import GEPAOptimizer, PromptWizardOptimizer, ProTeGi

MODEL = "openai/gpt-4o-mini"
seed = "Ticket: {ticket}\nQueue (RETURNS or REPAIRS):"
train = [
    {"ticket": "My new boots don't fit. Can I send them back?", "queue": "RETURNS"},
    {"ticket": "Can you fix a broken buckle on my pack?", "queue": "REPAIRS"},
]
mapper = BasicDataMapper({
    "response": "generated_output",
    "expected_response": "queue",
})
evaluator = Evaluator(metric=Equals(config={"case_sensitive": True}))
holdout = [
    {"ticket": "I ordered the wrong tent. Can I get a refund?", "queue": "RETURNS"},
    {"ticket": "Can you replace my sleeping bag zipper?", "queue": "REPAIRS"},
]


def accuracy(generator, rows):
    inputs = []
    for row in rows:
        reply = generator.generate({"ticket": row["ticket"]})
        if not reply:
            raise RuntimeError("No reply from the generator")
        inputs.append(mapper.map(reply, row))
    scores = evaluator.evaluate(inputs)
    return sum(item.score for item in scores) / len(rows)


teacher = LiteLLMGenerator(MODEL, "{prompt}")
shared = dict(
    evaluator=evaluator, data_mapper=mapper, dataset=train, initial_prompts=[seed]
)
protegi = ProTeGi(
    teacher, num_gradients=1, errors_per_gradient=1, prompts_per_gradient=1, beam_size=1
)
gepa = GEPAOptimizer(reflection_model=MODEL, generator_model=MODEL)
wizard = PromptWizardOptimizer(teacher, mutate_rounds=1, refine_iterations=1)
results = {
    "ProTeGi": protegi.optimize(**shared, num_rounds=1, eval_subset_size=len(train)),
    "GEPA": gepa.optimize(**shared, max_metric_calls=8),
    "PromptWizard": wizard.optimize(
        **shared, eval_subset_size=len(train),
        task_description="Return only RETURNS or REPAIRS. Keep {ticket} as the only placeholder.",
    ),
}
print(f"Baseline holdout: {accuracy(LiteLLMGenerator(MODEL, seed), holdout):.0%}")
for name, result in results.items():
    held_out = accuracy(result.best_generator, holdout)
    print(f"{name}: search={result.final_score:.0%}, holdout={held_out:.0%}")

## Read the result

`search` is each optimizer’s reported training score; `holdout` is accuracy on the same unseen tickets. One answer changes holdout accuracy by 50 percentage points. A tie is not evidence that the strategies are equivalent.